# 공공문서 RAG 검색 평가 실습

검색 평가에 집중할 수 있도록 PDF 로드, 메타데이터 정규화, 문서 분할 코드를 제공한다.

### 실습 목표

- 제공된 전처리 결과를 사용해 벡터 저장소를 구성한다.
- File Hit@k, Page Hit@k, MRR을 계산한다.
- Similarity Search와 MMR의 검색 성능을 비교한다.
- 실패 문항의 검색 결과를 확인하고 원인을 분석한다.

### 제공 데이터

- 검색 대상 문서: `data/public/*.pdf`
- 평가 데이터: `data/public/public_paragraph_golden_set.json`

golden set의 `question`은 검색 질의, `target_answer`는 기준 답변, `target_file_name`과 `target_page_no`는 정답 위치를 나타낸다.

`PyPDFLoader`가 만든 `source`에는 `data/public/파일명.pdf`처럼 경로가 포함될 수 있지만, golden set의 `target_file_name`에는 파일명만 들어 있다. 두 값을 바로 비교할 수 있도록 `Path(source).name`을 사용해 경로를 제거한다. 이 코드는 아래에 제공되어 있으므로 별도로 구현하지 않는다.

또한 `PyPDFLoader`의 `page`는 첫 페이지를 0으로 나타내지만 golden set의 `target_page_no`는 첫 페이지를 1로 나타낸다. 비교에 사용할 `page_no`에는 `page + 1`을 저장한다.

In [ ]:
import json
import time
from pathlib import Path
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

DATA_DIR = Path("data/public")
GOLDEN_SET_PATH = DATA_DIR / "public_paragraph_golden_set.json"
EMBEDDING_MODEL_NAME = "gemini-embedding-2"

### Golden set 확인

평가 문항 수와 필드 구성을 확인한다.

In [ ]:
eval_cases = json.loads(GOLDEN_SET_PATH.read_text(encoding="utf-8"))

print(f"평가 문항: {len(eval_cases)}개")
print(json.dumps(eval_cases[:5], ensure_ascii=False, indent=2))

### PDF 로드

모든 PDF를 페이지 단위로 불러온다. `PyPDFLoader.load()`의 결과에서 문서 하나는 PDF의 한 페이지에 해당한다.

불러온 페이지의 메타데이터는 다음과 같이 정리한다.

```python
{
    "source": "2024년 행정안전부 업무계획.pdf",
    "page": 0,
    "page_no": 1,
}
```

`source`는 golden set과 비교하기 위한 파일명이고, `page_no`는 golden set과 비교하기 위한 페이지 번호이다. 원래 로더가 제공한 `page`도 그대로 유지한다.

In [ ]:
page_documents = []

for pdf_path in sorted(DATA_DIR.glob("*.pdf")):
    pages = PyPDFLoader(pdf_path).load()

    for page in pages:
        # source에는 전체 경로 대신 golden set과 같은 파일명만 저장한다.
        page.metadata["source"] = Path(page.metadata["source"]).name
        # PyPDFLoader의 page는 0부터 시작하므로 사람이 보는 페이지 번호로 바꾼다.
        page.metadata["page_no"] = page.metadata["page"] + 1

    page_documents.extend(pages)

print(f"로드한 PDF: {len(list(DATA_DIR.glob('*.pdf')))}개")
print(f"로드한 페이지: {len(page_documents)}개")
print(page_documents[0].metadata)

### 문서 분할

페이지 정보를 유지한 채 문서를 청크로 분할한다. 이 실습에서는 제공된 전처리 결과를 사용한다.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100,
)

documents = splitter.split_documents(page_documents)

print(f"생성한 청크: {len(documents)}개")

### 벡터 저장소와 검색기 구성

동일한 문서로 Similarity Search와 MMR을 구성한다. 이 실습에서는 별도 저장 경로 없이 `Chroma.from_documents()`로 벡터 저장소를 만들고, 검색기 딕셔너리의 키는 `Similarity`와 `MMR`을 사용한다.

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL_NAME)

# TODO: vectorstore와 retrievers를 구성하세요.
vectorstore = None
retrievers = {}

### 평가 지표 구현

File Hit은 정답 파일의 검색 여부를, Page Hit은 정답 파일과 페이지가 함께 일치하는지를 확인한다. MRR은 최초로 등장한 정답 페이지 순위의 역수이다.

검색 결과 문서 `doc`과 golden set 문항 `case`는 다음 값으로 비교한다.

```python
파일 일치: doc.metadata["source"] == case["target_file_name"]
페이지 일치: doc.metadata["page_no"] == case["target_page_no"]
정답 페이지: 파일 일치 and 페이지 일치
```

예를 들어 정답 페이지가 검색 결과의 세 번째에 처음 등장하면 RR은 `1 / 3`이다. 상위 k개에 정답이 없으면 Hit은 0이고, 전체 검색 결과에 정답 페이지가 없으면 RR은 0.0이다.

In [ ]:
def file_hit_at_k(docs, case, k):
    # TODO
    pass


def page_hit_at_k(docs, case, k):
    # TODO
    pass


def reciprocal_rank(docs, case):
    # TODO
    pass

### 전체 평가

모든 질문을 검색하고 전략별 지표와 실행 시간을 기록한다. latency는 `retriever.invoke()` 한 번에 걸린 시간을 밀리초 단위로 측정한다. 실패 원인을 확인할 수 있도록 검색 결과의 순위, 파일명, 페이지 번호도 함께 저장한다.

In [ ]:
def evaluate_retriever(name, retriever, cases):
    rows = []
    # TODO: 평가 결과를 rows에 저장하세요.
    return rows


# TODO: 모든 검색 전략을 평가하세요.
evaluation_results = []
print(json.dumps(evaluation_results[:5], ensure_ascii=False, indent=2))

### 결과 비교

검색 전략별 평균 점수를 비교한다. 전략마다 문항 수 `count`와 `file_hit_1`, `page_hit_1`, `page_hit_3`, `page_hit_5`, `reciprocal_rank`, `latency_ms`의 평균을 계산한다. 여러 문항의 `reciprocal_rank` 평균이 MRR이다.

In [ ]:
# TODO: 검색 전략별 결과를 요약하세요.
summary = []
print(json.dumps(summary, ensure_ascii=False, indent=2))

### 실패 문항 분석

전체 평가 결과에서 `page_hit_5 == 0`인 행을 확인한다. 각 실패 행에 저장한 `retrieved`를 정답 파일·페이지와 비교하여 같은 파일의 다른 페이지가 검색되었는지, 다른 파일만 검색되었는지 분석한다.

In [ ]:
# TODO: 실패 문항을 추출하세요.
failures = []
print(json.dumps(failures, ensure_ascii=False, indent=2))

### 결과 정리

다음 내용을 작성한다.

- 가장 성능이 좋은 검색 전략과 판단 근거
- File Hit과 Page Hit 결과가 다른 이유
- 실패 문항에서 확인한 검색 결과의 특징
- 성능을 개선하기 위해 변경해 볼 설정